In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !uv add faiss-cpu

In [9]:
from langchain_classic.storage import LocalFileStore, InMemoryByteStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

In [3]:
embedding = OpenAIEmbeddings()

store = LocalFileStore("./cache/")

In [4]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,
)

d:\kbh\rag_one_2\rag_two\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [5]:
list(store.yield_keys())

[]

In [6]:
raw_documents = TextLoader("../data/appendix-keywords.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [7]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 219 ms
Wall time: 1.41 s


In [8]:
%time db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 15.6 ms
Wall time: 62.7 ms


In [10]:
store2 = InMemoryByteStore()

cached_embedder2 = CacheBackedEmbeddings.from_bytes_store(
    embedding, store2, namespace=embedding.model
)

In [11]:
%time db3 = FAISS.from_documents(documents, cached_embedder2)

CPU times: total: 31.2 ms
Wall time: 786 ms


In [12]:
%time db4 = FAISS.from_documents(documents, cached_embedder2)

CPU times: total: 0 ns
Wall time: 7.12 ms
